# 감염농장 전처리 파이프라인

가축질병 발생정보(감염농장)와 가금류 농장현황(농장현황)을 매칭해서 감염농장의 시군명·축종·상세구분·사육두수·좌표를 보완하고, `{시/도}_최종감염농장.csv`(12컬럼)를 만든다. 맨 위 `PROVINCE` 변수만 바꾸면 다른 시/도도 같은 코드로 처리된다.

**단계**: 인코딩 점검 → 시/도 선택 → 경로 설정 → 농장현황 생성 → 중복제거 → 감염농장 추출 → 감염농장 중복제거 → 주소 정확매칭 → 최종 조인

In [1]:
import glob
import os

DATA_DIR = "data"  # 이 노트북은 repo 루트에 있고, data/ 폴더를 가리킴

def _try_decode(raw, enc):
    try:
        raw.decode(enc)
        return True
    except (UnicodeDecodeError, LookupError):
        return False

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
broken = []
for path in csv_files:
    with open(path, "rb") as f:
        raw = f.read()
    if not _try_decode(raw, "utf-8"):
        broken.append(path)

print(f"전체 CSV {len(csv_files)}개 중 UTF-8로 읽히지 않는 파일: {len(broken)}개")

if broken:
    for path in broken:
        with open(path, "rb") as f:
            raw = f.read()
        fixed_text = None
        for enc in ("cp949", "euc-kr"):
            if _try_decode(raw, enc):
                fixed_text = raw.decode(enc)
                break
        if fixed_text is not None:
            with open(path, "w", encoding="utf-8-sig") as f:
                f.write(fixed_text)
            print(f"  복구 완료 (→ utf-8-sig): {path}")
        else:
            print(f"  ⚠️  자동 복구 실패, 직접 확인 필요: {path}")
else:
    print("✓ 모든 CSV가 정상적인 UTF-8 인코딩입니다. 깨진 파일 없음.")

전체 CSV 203개 중 UTF-8로 읽히지 않는 파일: 0개
✓ 모든 CSV가 정상적인 UTF-8 인코딩입니다. 깨진 파일 없음.


## 1. 처리할 시/도 선택 (이 변수만 바꿔서 재실행)

In [2]:
PROVINCE = "충청북도"  # 예: "경기도", "전라남도" — data/{PROVINCE}/ 폴더를 처리한다

## 2. 경로 설정 (시/도 폴더 안의 원본/파생 디렉토리 자동 탐지)

In [3]:
import glob
import os
import re
import csv
import difflib
import unicodedata
import pandas as pd
import numpy as np

PROVINCE = unicodedata.normalize("NFC", PROVINCE)  # macOS는 파일/폴더명을 NFD(분해형)로 저장해서 비교 전에 정규화 필요

PROV_DIR = os.path.join("data", PROVINCE)
RAW_INFECTION_DIR = os.path.join(PROV_DIR, "감염농장원본")
RAW_CENSUS_DIR = os.path.join(PROV_DIR, "농장현황원본")
CENSUS_DIR = os.path.join(PROV_DIR, "농장현황")
INFECTION_DIR = os.path.join(PROV_DIR, "감염농장")
os.makedirs(CENSUS_DIR, exist_ok=True)
os.makedirs(INFECTION_DIR, exist_ok=True)

assert os.path.isdir(RAW_CENSUS_DIR), f"{RAW_CENSUS_DIR} 폴더를 찾을 수 없음"
assert os.path.isdir(RAW_INFECTION_DIR), f"{RAW_INFECTION_DIR} 폴더를 찾을 수 없음"
print("원본 감염농장 폴더:", RAW_INFECTION_DIR)
print("원본 농장현황 폴더:", RAW_CENSUS_DIR)

원본 감염농장 폴더: data/충청북도/감염농장원본
원본 농장현황 폴더: data/충청북도/농장현황원본


## 3. 시군구_농장현황.csv 생성

두 가지 원본 파일 형식을 모두 지원한다:
- **날짜만 있는 파일명**(`YYYY-MM-DD.csv`, 예: 경기도): 전체 시군이 한 파일에 들어있고 컬럼명이 이미 통일되어 있음
- **`{시/도} {시군}_{설명}_{날짜}.csv`**(예: 전라남도): 파일 하나 = 시군 하나, 컬럼명이 파일마다 달라서 표준 스키마로 매핑·정규화함 (없는 컬럼은 빈 값). `(1)` 등 중복 다운로드로 보이는 파일은 스킵

In [4]:
# 시/도마다 원본 컬럼명이 제각각이라(농장명/사업장명/사업장명칭 등), 후보 목록 중 먼저 매칭되는 컬럼을 표준 컬럼으로 채택한다.
NAME_COLS = ["농장명", "사업장명", "사업장명칭", "축산업 사업장 명칭"]
SPECIES_COLS = ["축종", "주사육업종", "축종명", "등록축종"]
COUNT_COLS = ["사육두수(마리)", "사육두수", "사육수", "사육수수", "사육두수(규모)", "사육수(규모)", "사육수(마리)", "사육규모"]
ADDR_COLS = [
    "소재지지번주소", "사업장소재지", "상세주소", "주소", "농장주소",
    "사업장소재지(지번)", "사업장 소재지", "지번주소", "소재지전체주소",
]
ADDR_PART_COLS = ("소재지", "상세소재지")  # 주소가 두 컬럼(예: "소재지"+"상세소재지")으로 나뉘어 있는 경우 합쳐서 사용
DATE_ONLY_PATTERN = re.compile(r"^\d{4}-\d{2}-\d{2}$")
DATE_FORMATS = {4: "%Y", 6: "%Y%m", 8: "%Y%m%d"}  # 파일명 끝 숫자의 자릿수로 연/월/일 단위를 구분

# 행정구역 명칭이 바뀐 경우(예: 전라북도 -> 전북특별자치도) 주소에 옛 이름/새 이름이 섞여 나올 수 있어 별칭으로 등록
PROVINCE_ALIASES = {
    "전라북도": ["전북특별자치도"],
    "강원도": ["강원특별자치도"],
    "제주도": ["제주특별자치도"],
}


def pick_col(df, candidates):
    """후보 컬럼명 중 데이터프레임에 실제로 존재하는 첫 번째 컬럼을 반환."""
    return next((c for c in candidates if c in df.columns), None)


def has_province_prefix(addr):
    if addr.startswith(PROVINCE):
        return True
    return any(addr.startswith(alias) for alias in PROVINCE_ALIASES.get(PROVINCE, []))


def normalize_address(addr, sigun):
    """주소 앞에 '시/도 시군' 접두사가 없으면 채워서 다른 시군과 비교 가능한 형태로 통일."""
    addr = str(addr).strip()
    if not addr or addr.lower() == "nan":
        return np.nan
    if has_province_prefix(addr):
        return addr
    if sigun in addr:
        return f"{PROVINCE} {addr}"
    return f"{PROVINCE} {sigun} {addr}"


def read_census_csv(path, addr_cols=ADDR_COLS):
    """주소 컬럼에 따옴표 없는 쉼표가 섞여 칸이 한 개 더 생기는 경우(예: "316-3, 315-2")가 있어,
    pandas 기본 파서 대신 csv.reader로 직접 읽어 넘치는 칸을 주소 컬럼 위치에서 '/'로 합쳐 복구한다."""
    with open(path, encoding="utf-8-sig", newline="") as f:
        rows = list(csv.reader(f))
    header = [h.strip() for h in rows[0]]
    ncols = len(header)
    addr_idx = next((i for i, h in enumerate(header) if h in addr_cols), ncols - 1)

    fixed_rows = []
    for row in rows[1:]:
        excess = len(row) - ncols
        if excess > 0:
            row = row[:addr_idx] + ["/".join(row[addr_idx : addr_idx + excess + 1])] + row[addr_idx + excess + 1 :]
        elif excess < 0:
            row = row + [""] * (-excess)
        fixed_rows.append(row)

    return pd.DataFrame(fixed_rows, columns=header)


frames = []
skipped = []

for path in sorted(glob.glob(os.path.join(RAW_CENSUS_DIR, "*.csv"))):
    fname = unicodedata.normalize("NFC", os.path.basename(path))  # macOS NFD 파일명 정규화
    stem = os.path.splitext(fname)[0]

    if DATE_ONLY_PATTERN.match(stem):
        # 날짜만 있는 파일명: 전체 시군이 한 파일에 들어있고 컬럼명이 이미 통일된 형식 (예: 경기도)
        df = read_census_csv(path)
        df["조사날짜"] = pd.to_datetime(stem).date()
        frames.append(df)
        continue

    if re.search(r"\(\d+\)$", stem):
        # "(1)" 등으로 끝나는 파일은 중복 다운로드로 보고 스킵
        skipped.append(fname)
        continue

    # "{시/도} {시군}_{설명}_{날짜}.csv" 형식: 파일 하나 = 시군 하나, 컬럼명은 파일마다 다름
    tokens = re.split(r"[ _]+", stem)
    sigun = tokens[1] if len(tokens) > 1 else None
    date_hits = [d for d in re.findall(r"\d+", stem) if len(d) in DATE_FORMATS]
    if sigun is None or not date_hits:
        print(f"  ⚠️  파일명에서 시군/날짜 추출 실패: {fname}")
        continue
    date_str = date_hits[-1]
    survey_date = pd.to_datetime(date_str, format=DATE_FORMATS[len(date_str)]).date()

    df = read_census_csv(path)

    name_col = pick_col(df, NAME_COLS)
    species_col = pick_col(df, SPECIES_COLS)
    count_col = pick_col(df, COUNT_COLS)
    addr_col = pick_col(df, ADDR_COLS)

    out = pd.DataFrame(index=df.index)
    out["시군명"] = sigun
    out["농장명"] = df[name_col] if name_col else np.nan
    out["축종명"] = df[species_col] if species_col else np.nan
    out["상세구분"] = np.nan  # 전라남도 등에는 산란계/육계 같은 상세구분 컬럼이 없어 항상 빈 값

    if count_col:
        cleaned = df[count_col].astype(str).str.replace(",", "", regex=False).str.strip()
        out["사육두수(마리)"] = pd.to_numeric(cleaned, errors="coerce")
    else:
        out["사육두수(마리)"] = np.nan

    if addr_col:
        raw_addr = df[addr_col]
    elif all(c in df.columns for c in ADDR_PART_COLS):
        raw_addr = df[ADDR_PART_COLS[0]].astype(str).str.strip() + " " + df[ADDR_PART_COLS[1]].astype(str).str.strip()
    else:
        raw_addr = pd.Series([np.nan] * len(df), index=df.index)

    out["소재지지번주소"] = raw_addr.apply(lambda a: normalize_address(a, sigun))
    out["WGS84위도"] = np.nan  # 좌표는 경기도 외에는 원본에 없음
    out["WGS84경도"] = np.nan
    out["조사날짜"] = survey_date
    frames.append(out)

if skipped:
    print(f"중복 다운로드로 추정되어 스킵한 파일 {len(skipped)}개: {skipped}")

census_all = pd.concat(frames, ignore_index=True)

for sigun, group in census_all.groupby("시군명"):
    group = group.sort_values("농장명").reset_index(drop=True)
    out_path = os.path.join(CENSUS_DIR, f"{sigun}_농장현황.csv")
    group.to_csv(out_path, index=False, encoding="utf-8-sig")

print(f"✓ {census_all['시군명'].nunique()}개 시군구 농장현황 파일 생성 완료 -> {CENSUS_DIR}")

✓ 2개 시군구 농장현황 파일 생성 완료 -> data/충청북도/농장현황


## 4. 농장현황 중복 제거 (시군명·농장명·상세구분·소재지지번주소·위도·경도·사육두수 동일 시)

In [5]:
DEDUP_COLS = ["시군명", "농장명", "상세구분", "소재지지번주소", "WGS84위도", "WGS84경도", "사육두수(마리)"]

dedup_count = 0
for census_file in glob.glob(os.path.join(CENSUS_DIR, "*_농장현황.csv")):
    df = pd.read_csv(census_file, encoding="utf-8-sig")
    before = len(df)
    df = df.drop_duplicates(subset=DEDUP_COLS, keep="first")
    after = len(df)
    dedup_count += (before - after)
    df.to_csv(census_file, index=False, encoding="utf-8-sig")
    if before > after:
        print(f"  {os.path.basename(census_file)}: {before} → {after} (-{before-after}건)")

print(f"\n✓ 총 {dedup_count}건 중복 제거 완료")

  음성군_농장현황.csv: 583 → 229 (-354건)

✓ 총 354건 중복 제거 완료


## 5. 감염농장 추출 (FARM_NM, FARM_LOCPLC, OCCRRNC_DE, LVSTCKSPC_CODE만 추출, FARM_LOCPLC 주소로 시군명 자동 분할)

In [6]:
INFECTION_COLS = ["FARM_NM", "FARM_LOCPLC", "OCCRRNC_DE", "LVSTCKSPC_CODE"]

raw_frames = []
for path in glob.glob(os.path.join(RAW_INFECTION_DIR, "*.csv")):
    df = pd.read_csv(path, encoding="utf-8-sig", on_bad_lines="skip")
    if all(col in df.columns for col in INFECTION_COLS):
        raw_frames.append(df[INFECTION_COLS])
    else:
        print(f"  ⚠️  {os.path.basename(path)}: 필요한 컬럼 없음")

infection_all = pd.concat(raw_frames, ignore_index=True)

# 원본 API 일부 값에 줄바꿈이 그대로 박혀있어(예: "아름농장\r\n") CSV로 다시 저장하면
# 멀티라인 셀이 되어 엑셀/뷰어에서 행이 깨져 보임 -> 줄바꿈을 제거해서 한 줄로 정리
for col in ("FARM_NM", "FARM_LOCPLC"):
    infection_all[col] = infection_all[col].astype(str).str.replace(r"[\r\n]+", "", regex=True).str.strip()

infection_all["시군명"] = infection_all["FARM_LOCPLC"].astype(str).str.split().str[1]

infection_count = 0
for sigun, group in infection_all.groupby("시군명"):
    out_path = os.path.join(INFECTION_DIR, f"{sigun}_감염농장.csv")
    group[INFECTION_COLS].to_csv(out_path, index=False, encoding="utf-8-sig")
    infection_count += len(group)
    print(f"  {sigun}: {len(group)}건")

print(f"\n✓ 총 {infection_count}건 감염농장 데이터 추출 완료 -> {INFECTION_DIR}")

  괴산군: 7건
  영동군: 1건
  옥천군: 2건
  음성군: 140건
  증평군: 2건
  진천군: 55건
  청원군: 1건
  청주시: 15건
  충주시: 6건

✓ 총 229건 감염농장 데이터 추출 완료 -> data/충청북도/감염농장


## 6. 감염농장 중복 제거 (위 4개 컬럼 모두 동일 시 — 원본 API에 같은 사건이 다른 발생번호로 중복 등록된 경우)

In [7]:
infection_dedup_count = 0
for infection_path in glob.glob(os.path.join(INFECTION_DIR, "*_감염농장.csv")):
    df = pd.read_csv(infection_path, encoding="utf-8-sig")
    before = len(df)
    df = df.drop_duplicates(subset=INFECTION_COLS, keep="first")
    after = len(df)
    if before > after:
        print(f"  {os.path.basename(infection_path)}: {before} → {after} (-{before-after}건)")
        infection_dedup_count += (before - after)
    df.to_csv(infection_path, index=False, encoding="utf-8-sig")

print(f"\n✓ 감염농장 데이터 총 {infection_dedup_count}건 중복 제거 완료")

  음성군_감염농장.csv: 140 → 138 (-2건)
  진천군_감염농장.csv: 55 → 54 (-1건)

✓ 감염농장 데이터 총 3건 중복 제거 완료


## 7. 주소 정확매칭 (FARM_NM == 농장명이면 FARM_LOCPLC를 소재지지번주소로 덮어씀, 미매칭이면 원본 유지)

In [8]:
address_updated = 0
address_unmatched = 0

for infection_path in glob.glob(os.path.join(INFECTION_DIR, "*_감염농장.csv")):
    sigun = os.path.splitext(os.path.basename(infection_path))[0].replace("_감염농장", "")
    census_path = os.path.join(CENSUS_DIR, f"{sigun}_농장현황.csv")

    if not os.path.exists(census_path):
        print(f"  ⚠️  {sigun}: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)")
        continue

    infection_df = pd.read_csv(infection_path, encoding="utf-8-sig")
    census_df = pd.read_csv(census_path, encoding="utf-8-sig")

    infection_df["FARM_NM"] = infection_df["FARM_NM"].astype(str)
    census_df["농장명"] = census_df["농장명"].astype(str)

    # 지번주소가 있는 농장만 매칭 대상으로 사용, 동명 농장이 여러 건이면 조사날짜가 가장 최신인 것을 사용
    addr_lookup = (
        census_df.dropna(subset=["소재지지번주소"])
        .sort_values("조사날짜", ascending=False)
        .drop_duplicates(subset="농장명", keep="first")
        .set_index("농장명")["소재지지번주소"]
    )

    matched_mask = infection_df["FARM_NM"].isin(addr_lookup.index)
    infection_df.loc[matched_mask, "FARM_LOCPLC"] = infection_df.loc[matched_mask, "FARM_NM"].map(addr_lookup)

    address_updated += int(matched_mask.sum())
    address_unmatched += int((~matched_mask).sum())

    infection_df.to_csv(infection_path, index=False, encoding="utf-8-sig")

print(f"✓ FARM_LOCPLC 덮어쓰기 완료: 매칭(주소 덮어씀) {address_updated}건, 미매칭(원본 유지) {address_unmatched}건")

  ⚠️  영동군: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)
  ⚠️  옥천군: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)
  ⚠️  청주시: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)
  ⚠️  청원군: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)
  ⚠️  괴산군: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)
  ⚠️  증평군: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)
  ⚠️  충주시: 농장현황 파일 없음 (주소 덮어쓰기 건너뜀)
✓ FARM_LOCPLC 덮어쓰기 완료: 매칭(주소 덮어씀) 37건, 미매칭(원본 유지) 155건


## 8. 최종 조인 → `최종감염농장/{PROVINCE}_최종감염농장.csv` (정확매칭 → 유사매칭(≥60%) → 주소기반 시군명/농장명 보완, 동명 농장은 조사날짜 최신 우선, 매칭된 행은 FARM_LOCPLC도 소재지지번주소로 덮어씀, 12컬럼)

In [9]:
def addr_key(addr, n=4):
    return tuple(str(addr).split()[:n])

def name_similarity(a, b):
    return difflib.SequenceMatcher(None, str(a), str(b)).ratio()

SIMILARITY_THRESHOLD = 0.6
FARM_COLS = ["시군명", "농장명", "축종명", "상세구분", "사육두수(마리)", "소재지지번주소", "WGS84위도", "WGS84경도"]
INFECTION_JOIN_COLS = ["FARM_NM", "FARM_LOCPLC", "OCCRRNC_DE", "LVSTCKSPC_CODE"]
FINAL_COLS = FARM_COLS + INFECTION_JOIN_COLS
FINAL_DIR = "최종감염농장"
os.makedirs(FINAL_DIR, exist_ok=True)

final_rows = []
exact_count = 0
fuzzy_count = 0
fallback_count = 0
locplc_overwritten = 0

# 감염농장을 기준으로 순회해야 한다 (농장현황 기준으로 돌면 농장현황 자체가 없는
# 시군의 감염농장이 통째로 누락됨). 농장현황이 없으면 주소 기반 보완만 적용.
for infection_file in sorted(glob.glob(os.path.join(INFECTION_DIR, "*_감염농장.csv"))):
    sigun = os.path.splitext(os.path.basename(infection_file))[0].replace("_감염농장", "")
    census_file = os.path.join(CENSUS_DIR, f"{sigun}_농장현황.csv")

    try:
        infection_df = pd.read_csv(infection_file, encoding="utf-8-sig")
        infection_df["FARM_NM"] = infection_df["FARM_NM"].astype(str)

        census_by_name = None
        addr_groups = {}
        if os.path.exists(census_file):
            census_df = pd.read_csv(census_file, encoding="utf-8-sig")
            if all(col in census_df.columns for col in FARM_COLS):
                census_df["농장명"] = census_df["농장명"].astype(str)
                # 동명 농장이 여러 건이면 조사날짜가 가장 최신인 행만 사용
                census_by_name = (
                    census_df.sort_values("조사날짜", ascending=False)
                    .drop_duplicates(subset="농장명", keep="first")
                    .set_index("농장명", drop=False)
                )
                census_df["_addr_key"] = census_df["소재지지번주소"].apply(addr_key)
                addr_groups = census_df.groupby("_addr_key")["농장명"].apply(list).to_dict()
            else:
                print(f"  ⚠️  {sigun} 농장현황: 필요한 컬럼 부재")
        else:
            print(f"  ⚠️  {sigun}: 농장현황 파일 없음 (주소 기반으로만 보완)")

        sigun_exact = sigun_fuzzy = sigun_fallback = 0

        for _, irow in infection_df.iterrows():
            farm_nm = irow["FARM_NM"]
            locplc = irow["FARM_LOCPLC"]
            match_row = None

            if census_by_name is not None and farm_nm in census_by_name.index:
                match_row = census_by_name.loc[farm_nm]
                sigun_exact += 1
            elif census_by_name is not None:
                candidates = addr_groups.get(addr_key(locplc), [])
                best_name, best_score = None, 0.0
                for c in candidates:
                    score = name_similarity(farm_nm, c)
                    if score > best_score:
                        best_name, best_score = c, score
                if best_name is not None and best_score >= SIMILARITY_THRESHOLD:
                    match_row = census_by_name.loc[best_name]
                    sigun_fuzzy += 1

            if match_row is not None:
                record = {col: match_row[col] for col in FARM_COLS}
            else:
                sigun_fallback += 1
                record = {col: np.nan for col in FARM_COLS}
                tokens = str(locplc).split()
                record["시군명"] = tokens[1] if len(tokens) > 1 else np.nan
                record["농장명"] = farm_nm

            for col in INFECTION_JOIN_COLS:
                record[col] = irow[col]

            # 정확매칭이든 유사매칭이든 농장현황과 매칭됐으면, 더 정확한 소재지지번주소로
            # FARM_LOCPLC도 덮어쓴다 (과거엔 정확매칭만 덮어써서 유사매칭 행은 jibun_address와
            # FARM_LOCPLC가 서로 다른 주소로 남는 불일치가 있었음)
            if match_row is not None and pd.notna(record["소재지지번주소"]):
                if record["FARM_LOCPLC"] != record["소재지지번주소"]:
                    locplc_overwritten += 1
                record["FARM_LOCPLC"] = record["소재지지번주소"]

            final_rows.append(record)

        exact_count += sigun_exact
        fuzzy_count += sigun_fuzzy
        fallback_count += sigun_fallback

        print(f"  {sigun}: {len(infection_df)}건 (정확매칭 {sigun_exact}, 유사매칭 {sigun_fuzzy}, 미매칭/주소보완 {sigun_fallback})")

    except Exception as e:
        print(f"  ⚠️  {sigun}: {str(e)}")

# 최종 파일 생성
if final_rows:
    final_df = pd.DataFrame(final_rows, columns=FINAL_COLS)
    out_path = os.path.join(FINAL_DIR, f"{PROVINCE}_최종감염농장.csv")
    final_df.to_csv(out_path, index=False, encoding="utf-8-sig")

    total = exact_count + fuzzy_count + fallback_count
    print(f"\n✓ 최종 파일 생성 완료: {out_path}")
    print(f"  - 총 행: {len(final_df)}")
    print(f"  - 컬럼: {len(final_df.columns)} -> {list(final_df.columns)}")
    print(f"  - 정확매칭: {exact_count}건")
    print(f"  - 유사매칭(>={int(SIMILARITY_THRESHOLD*100)}%): {fuzzy_count}건")
    print(f"  - 미매칭(주소 기반 시군명/농장명만 채움): {fallback_count}건")
    print(f"  - FARM_LOCPLC를 소재지지번주소로 덮어쓴 행: {locplc_overwritten}건")
    if total:
        print(f"  - 매칭률(정확+유사): {(exact_count+fuzzy_count)*100/total:.1f}%")
else:
    print("⚠️  최종 데이터 없음")


  ⚠️  괴산군: 농장현황 파일 없음 (주소 기반으로만 보완)
  괴산군: 7건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 7)
  ⚠️  영동군: 농장현황 파일 없음 (주소 기반으로만 보완)
  영동군: 1건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 1)
  ⚠️  옥천군: 농장현황 파일 없음 (주소 기반으로만 보완)
  옥천군: 2건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 2)
  음성군: 138건 (정확매칭 30, 유사매칭 5, 미매칭/주소보완 103)
  ⚠️  증평군: 농장현황 파일 없음 (주소 기반으로만 보완)
  증평군: 2건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 2)
  진천군: 54건 (정확매칭 7, 유사매칭 2, 미매칭/주소보완 45)
  ⚠️  청원군: 농장현황 파일 없음 (주소 기반으로만 보완)
  청원군: 1건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 1)
  ⚠️  청주시: 농장현황 파일 없음 (주소 기반으로만 보완)
  청주시: 15건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 15)
  ⚠️  충주시: 농장현황 파일 없음 (주소 기반으로만 보완)
  충주시: 6건 (정확매칭 0, 유사매칭 0, 미매칭/주소보완 6)

✓ 최종 파일 생성 완료: 최종감염농장/충청북도_최종감염농장.csv
  - 총 행: 226
  - 컬럼: 12 -> ['시군명', '농장명', '축종명', '상세구분', '사육두수(마리)', '소재지지번주소', 'WGS84위도', 'WGS84경도', 'FARM_NM', 'FARM_LOCPLC', 'OCCRRNC_DE', 'LVSTCKSPC_CODE']
  - 정확매칭: 37건
  - 유사매칭(>=60%): 7건
  - 미매칭(주소 기반 시군명/농장명만 채움): 182건
  - FARM_LOCPLC를 소재지지번주소로 덮어쓴 행: 7건
  - 매칭률(정확+유사): 19.5%
